In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.algorithms import VQC
from qiskit_ibm_runtime import QiskitRuntimeService, Session, SamplerV2 as Sampler
from qiskit_algorithms.optimizers import COBYLA

In [2]:
## Usando Dados Credit-G para teste
data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = data.frame

In [3]:
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


In [4]:
## Selecionando 4 colunas para definir o risco (4 qubits)
X = data.frame[['duration', 'credit_amount', 'age', 'residence_since']].values
y = data.frame['class'].values.reshape(-1, 1)

In [5]:
# OH VQC exige 2 saídas para classes bom e ruim
encoder = OneHotEncoder(sparse_output=False)
y_oh = encoder.fit_transform(y)

In [6]:
## Normalização
X_scaled = StandardScaler().fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=108)

In [8]:
from credentials import token
# Seletor de servidor, "least_busy"
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
print(f'Conectado: {backend.name}')

qiskit_runtime_service._discover_account:WARNING:2026-04-14 10:42:03,750: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-04-14 10:42:07,686: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Learning. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-14 10:42:08,417: Loading instance: Learning, plan: open
qiskit_runtime_service.backends:WARNING:2026-04-14 10:42:10,964: Using instance: Learning, plan: open


Conectado: ibm_fez


In [9]:
num_qubits = X_train.shape[1]

In [10]:
### Transformando dados em inferencia quantica
f_map = zz_feature_map(feature_dimension=num_qubits, reps=2)

In [11]:
## Ajuste dos pesos
ansatz = real_amplitudes(num_qubits=num_qubits, reps=3)

In [12]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
f_map_t = pm.run(f_map)
ansatz_t = pm.run(ansatz)

In [ ]:
sampler = Sampler(mode=backend)
## treinando o modelo
vqc = VQC(
    feature_map = f_map_t,
    ansatz=ansatz_t,
    optimizer=COBYLA(maxiter=20),
    sampler=sampler
)
print('Iniciando treinamento no processador quântico...')
vqc.fit(X_train, y_train)

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Iniciando treinamento no processador quântico...


In [ ]:
score = vqc.score(X_test, y_test)
print(f"Acurácia Quântica: {score * 100:.2f}%")